
# QUASE: Automated Quality Assessment and Enhancement of User Stories Using Multi-Agents Pipeline



In [ ]:
import json
import requests
import time
import re

##QUASE-Assess

In [ ]:
QUS_EVALUATION_SYSTEM_PROMPT = """You are an expert in agile requirements engineering and user story quality assessment. Your task is to evaluate user stories against the Quality User Story (QUS) framework, which consists of 13 criteria organized into three categories: Syntactic, Semantic, and Pragmatic quality.

You will receive either:
1. A SET of user stories for individual criteria evaluation (criteria 1-6, 8-9)
2. A SET of user stories for set-level criteria evaluation (criteria 7, 10-13)

Evaluate rigorously and objectively based ONLY on the definitions provided. Follow the step-by-step detection strategies for each criterion."""

QUS_INDIVIDUAL_EVALUATION_PROMPT = """
Evaluate this SET of user stories against the individual QUS quality criteria:

USER STORIES:
{user_stories_list}


---


EVALUATION CRITERIA:

## SYNTACTIC QUALITY (Structure without meaning)

### 1. Well-formed
**Definition:** A user story includes at least a role and a means.
**Standard template:** "As a <role>, I want <means>, [so that <end>]"

**Evaluation:**
- Does it include a role (who)?
- Does it include a means (what)?
- Score 0 if BOTH present, 1 if either missing

---

### 2. Atomic
**Definition:** A user story expresses a requirement for exactly one feature.

**DETECTION STRATEGY - Follow these steps:**

**STEP 1: Scan the MEANS for conjunctions**
- Search for: "and", "or", "&", "/"
- Flag each occurrence for review

**STEP 2: For each conjunction found, check BOTH sides:**
- Left side: Does it describe a distinct feature/action/component?
- Right side: Does it describe a distinct feature/action/component?
- If BOTH sides are features/actions/components → VIOLATION (Score 0)

**STEP 3: Common violation patterns:**
- x "AI **and** real-time data" (two data sources → NOT atomic)
- x "sensors **and** algorithms" (two components → NOT atomic)
- x "optimize **and** reduce" (two separate goals → NOT atomic)
- x "regulations **and** safety standards" (two deliverables → NOT atomic)
- x "control data **and ensure** security" (two separate actions → NOT atomic)
- x "login **or** register" (two distinct operations → NOT atomic)

**Domain-aware exceptions (NOT violations):**
- ✓ "bread and butter" (idiom, single concept)
- ✓ "terms and conditions" (inseparable legal pair)
- ✓ "Research and Development" (single department name)
- ✓ "salt and pepper" (conventional pair treated as unit)

**Compound nouns (NOT violations):**
- ✓ "peanut butter and jelly sandwich" (single item)
- ✓ "AI-powered recommendation engine" (single feature)

**Adjective pairs (NOT violations):**
- ✓ "fast and efficient" (adjectives modifying same feature)
- ✓ "safe and reliable" (quality attributes of single feature)

**Evaluation Process:**
1. Found conjunction in means? → Check both sides
2. Both sides = distinct features/actions/components? → Score 1 (NOT atomic)
3. No conjunction OR conjunction is exception/adjective pair? → Score 0 (atomic)

---

### 3. Minimal
**Definition:** A user story contains nothing more than role, means, and ends.

**Violations:**
- Implementation hints, technical details beyond feature description
- UI mockup references, design specifications
- Testing instructions, notes in parentheses
- Comments that belong in description/acceptance criteria
- Multiple sentences with extra context

**Evaluation:**
- Does it contain ONLY role, means, and optional ends?
- Is there text in parentheses, brackets, or after dashes/semicolons?
- Are there implementation details that should be in comments?
- Score 0 if minimal, 1 if extra information present



---

## SEMANTIC QUALITY (Meaning and relationships)

### 4. Conceptually sound
**Definition:** The means expresses a CONCRETE FEATURE and the ends expresses a RATIONALE (not anything else).

**PART 1: Is the MEANS a concrete feature?**

**Concrete Features (✓ Valid means):**
- Specific functionality: "navigation system", "payment processing", "user authentication"
- Observable capability: "detect obstacles", "optimize routes", "send notifications"
- Deliverable output: "generate report", "display dashboard", "create invoice"
- Tangible system behavior: "automatically save drafts", "validate input", "filter results"

**NOT Features (✗ Invalid means):**
- x Vague exploration: "explore", "investigate", "research", "study", "consider"
- x Meta-activities: "analyze options", "evaluate alternatives", "assess feasibility"
- x Process actions: "define requirements", "conduct interviews", "gather feedback"
- x Placeholder intent: "look into", "figure out", "work on", "think about"

**Detection Strategy for Means:**
1. Identify the main verb in the means
2. Ask: "Can this be IMPLEMENTED and TESTED in a sprint?"
   - ✓ "develop navigation system" → YES (concrete deliverable)
   - ✗ "explore navigation options" → NO (research activity)
3. Ask: "Does this produce WORKING SOFTWARE or a CONCRETE DELIVERABLE?"
   - ✓ "payment processing feature" → YES
   - ✗ "investigate payment providers" → NO

**PART 2: Does the ENDS express rationale (WHY)?**

**Valid Ends (✓ Rationale):**
- User benefit: "so that I can save time"
- Business value: "so that I improve customer satisfaction"
- Personal goal: "so that I reduce costs"
- Motivation: "so that I feel more secure"

**Invalid Ends (✗ NOT rationale):**
- x Hidden feature: "so that I can see landmark locations" (this is ANOTHER feature, needs separate story)
- x Implementation detail: "so that the system uses caching" (HOW, not WHY)
- x Dependency on other functionality: "so that other features can work" (implies missing story)
- x Technical explanation: "so that the database stays synchronized" (implementation concern)

**Examples:**

✓ CONCEPTUALLY SOUND:
- Means: "an AV that can navigate me around town" (concrete feature)
- Ends: "giving me greater independence" (rationale - WHY)

✗ NOT CONCEPTUALLY SOUND:
- Means: "explore the use of AVs for delivery" (exploration activity, NOT a feature)
- Should be: "a delivery routing system using AVs" (concrete feature)

✗ NOT CONCEPTUALLY SOUND:
- Means: "update user profile"
- Ends: "so that I can see profile pictures" (this is ANOTHER feature - viewing pictures, not rationale)

**Evaluation:**
1. Score 1 if means is exploration/meta-activity (not a concrete feature)
2. Score 1 if ends describes another feature (dependency issue)
3. Score 1 if ends describes implementation (should be in technical notes)
4. Score 0 only if means = concrete feature AND ends (if present) = rationale for WHY OR a general goal or aspiration

---

### 5. Problem-oriented
**Definition:** A user story only specifies the problem (WHAT is needed), not the solution (HOW to build it).

**Evaluation:**
- Does it describe WHAT feature is needed? -> Problem (Score 0)
- Does it prescribe HOW to build it with specific implementation OR What specific technology to use? -> Solution (Score 1)
- General technology mention ONLY. no details allowed -> Problem (Score 0)
- If specific tech/brand/algorithm/UI placement is mentioned -> Score 1.

---

### 6. Unambiguous
**Definition:** A user story avoids terms or abstractions that lead to multiple interpretations.

**Violations:**
- Vague terms without specification: "content", "data", "information", "stuff", "things"
- Ambiguous pronouns: "it", "this", "that", "them" without clear referent
- Abstract concepts: "efficient", "fast", "good", "appropriate" without measurable criteria
- Superclass terms: "vehicle" when you mean "car", "content" when you mean "video"
- Context-dependent words: "soon", "large", "many", "frequently" without quantification
- Unclear scope: "manage users" (create? edit? delete? all of above?)

**Acceptable:**
- Specific terms: "video", "user profile", "payment transaction"
- Domain terminology: "autonomous vehicle", "navigation system"
- Clear referents: "the uploaded image", "my shopping cart"
- Quantified when possible: "within 2 seconds", "at least 10 items"

**Evaluation:**
- Are all terms clear and specific?
- Could different developers interpret this differently?
- Are domain terms used consistently?
- Score 0 if unambiguous, 1 if terms could mean multiple things

---

## PRAGMATIC QUALITY (Audience interpretation)

### 8. Full sentence
**Definition:** A user story is a well-formed, grammatically correct full sentence.

**Violations:**
- Sentence fragments: "Login functionality", "User profile update"
- Missing punctuation
- Grammatical errors
- Not following standard sentence structure

**Evaluation:**
- Is it a grammatically correct, complete sentence?
- Does it have proper capitalization and punctuation?
- Score 0 if yes, 1 if no

---

### 9. Estimatable
**Definition:** A story does not denote a coarse-grained requirement that is difficult to plan and prioritize.

**Violations:**
- Too large/vague to estimate effort accurately
- Unclear scope - could be 1 day or 1 month of work
- Missing critical details needed for sizing
- Epic-sized story not broken down
- Exploratory stories: "explore options", "investigate solutions"
- Multiple features bundled together making size unclear

**Acceptable:**
- Clear, bounded scope
- Single feature with understood complexity
- Enough detail for developers to estimate effort
- Can reasonably complete in one sprint

**Evaluation:**
- Can a developer reasonably estimate the effort required?
- Is the scope clear enough for sprint planning?
- Could this be completed in a single sprint (1-3 weeks)?
- Score 1 if estimatable, 0 if too large/vague/undefined


---

OUTPUT FORMAT:
Return your evaluation as JSON with this exact structure:

{{

  "well_formed": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},
  "atomic": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},
  "minimal": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},
  "conceptually_sound": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},
  "problem_oriented": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},
  "unambiguous": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},
  "full_sentence": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},
  "estimatable": {{
    "US1": {{"score": 1, "reason": "brief"}},
    "US2": {{"score": 0, "reason": "brief"}},
    "US3": {{"score": 1, "reason": "brief"}},
    "US4": {{"score": 1, "reason": "brief"}},
    "US5": {{"score": 1, "reason": "brief"}},
    "US6": {{"score": 0, "reason": "brief"}},
    "US7": {{"score": 1, "reason": "brief"}},
    "US8": {{"score": 0, "reason": "brief"}},
    "US9": {{"score": 1, "reason": "brief"}},
    "US10": {{"score": 1, "reason": "brief"}},
  }},

  "total_score": 24,
  "quality_issues": ["list of specific problems found"],
  "improvement_suggestions": ["specific suggestions to fix issues"]
}}
"""

QUS_SET_EVALUATION_PROMPT =  """
Evaluate this SET of user stories against the QUS quality criteria:

USER STORIES:
{user_stories_list}

---

EVALUATION CRITERIA:


## SEMANTIC QUALITY (Set-level)
### 7. Conflict-free
**Definition:** A user story should not be inconsistent with any other user story.
**Violations:**
- Story A allows editing "any landmark" while Story B only allows editing "landmarks I added"
- Contradictory business rules across stories
- Inconsistent access control or permissions
- Direct Contradiction
- Goal Conflicts
- Resource Conflicts

-  Cross-Cutting Concerns (Examples)**
- **Data collection** (needs data) vs **Privacy protection** (minimize data)
- **System autonomy** (AI decides) vs **Security controls** (human oversight)
- **Performance optimization** (speed) vs **Safety regulations** (compliance checks)
- **Functionality** (features) vs **Regulatory compliance** (restrictions)
- **Innovation** (new capabilities) vs **Risk mitigation** (conservative approach)


**Evaluation:**
- Identify pairs of stories that contradict each other
- Return list of conflicting pairs with explanation
---

## PRAGMATIC QUALITY (Set-level)

### 10. Unique
**Definition:** Every user story is unique, duplicates are avoided.

 Check each user story against ALL other stories in the set using these predicates:

**TYPE 1: Full Duplicate (Exact Match)**
- **Detection**: Compare entire story texts character-by-character (case-insensitive, ignore whitespace)
- **Evaluation**:  "exact duplicate"

**TYPE 2: Semantic Duplicate (Same Meaning, Different Wording)**
- **Detection**: If role + action + object are semantically equivalent → SEMANTIC DUPLICATE
- **Evaluation**:  "semantic duplicate"

**TYPE 3: Different Means, Same End (Feature Variation or Conflict)**
- **Detection**: If means are different BUT ends overlap/match
- **Evaluation**: "overlapping ends"

**TYPE 4: Same Means, Different End (Multiple Rationales)**
- **Detection**:If means are semantically equivalent,AND ends are different
- **Evaluation**: "same feature, different rationale"

**TYPE 5: Different Role, Same Means/End (Usually ACCEPTABLE)**
- **Detection**: different role, same story
- **Evaluation**: "diffRoleSameStory"

**TYPE 6: End = Means (purposeIsMeans Relationship)**
- **Detection Steps**:
  1. Extract end(s) from story λ1
  2. Extract means from ALL other stories
  3. Check if λ1's end matches λ2's means semantically
  4. If YES -> λ1's rationale is actually λ2's feature
- **Evaluation**: Flag as "purposeIsMeans" - indicates end should be separate story, violates both uniqueness AND independence

---

### 11. Uniform
**Definition:** All user stories in a specification employ the same template.

**Check for consistency in:**
- Role format: "As a" vs "As an" (grammatically correct usage is OK)
- Means format: "I want" vs "I want to" vs "I am able to" vs "I can" vs "I need"
- Ends format: "so that" vs "in order to" vs "so" vs no ends marker
- Punctuation: commas after role, periods at end

**Violations:**
- Mixing different means indicators inconsistently
- Some stories have ends, others don't (when context suggests they should)
- Different sentence structures across stories
- Inconsistent punctuation patterns

**Evaluation:**
- Identify the most common template format used
- Count stories that deviate from this standard
- List specific deviations with story IDs
- Consider: Is deviation grammatically necessary or just inconsistent?

---

### 12. Independent
**Definition:** The user story is self-contained and has no inherent dependencies on other stories.

**Dependency Types:**

**Type 1: Causal Dependencies (CRUD chains)**
- "Create X" must happen before "Edit X" or "Delete X"
- "Register user" before "User login"
- "Upload file" before "Download file"
- Note: These are often unavoidable but should be made explicit

**Type 2: Functional Dependencies**
- Story A requires functionality from Story B to work
- "View profile picture" depends on "Upload profile picture"
- "Send notification" depends on "Configure notification settings"

**Type 3: Data Dependencies**
- Story references data that another story creates
- "Generate report from survey data" depends on "Collect survey responses"

**Type 4: Is-A Dependencies (Superclass/Subclass)**
- Story about "content" depends on stories defining what content types exist
- "Manage vehicles" depends on stories that define vehicle types

**Evaluation:**
- Identify dependencies between stories
- Classify dependency type
- Note: Some dependencies are unavoidable (CRUD operations)
- Recommend making dependencies explicit in backlog
- Flag dependencies that could be eliminated by story restructuring

---

### 13. Complete
**Definition:** Implementing a set of user stories creates a feature-complete application, no steps are missing.

**Check for Missing Stories:**

**Missing CRUD operations:**
- Can Create but no Read/View story
- Can Update but no Create story
- Can Delete but no way to Create first

**Missing user workflows:**
- Story for outcome but not for setup
- Story for action but not for prerequisites
- Gaps in user journey (e.g., login → dashboard, but no navigation between)

**Missing cross-cutting concerns:**
- Functional stories but no error handling
- Features but no security/authentication
- Data collection but no data management/deletion

**Referenced but undefined features:**
- Story mentions "user profile" but no story creates profiles
- Story assumes "dashboard" exists but no story defines it
- Story refers to "settings" with no story for configuration

**Evaluation:**
- Identify missing stories implied by existing ones
- Look for incomplete CRUD chains
- Check for missing supporting functionality
- List specific missing stories with rationale
- Relate missing stories to existing story IDs

---

OUTPUT FORMAT:
Return your evaluation as JSON with this exact structure:

{{

  "conflict_free": {{
    "has_conflicts": true,
    "conflicts": [
      {{"story_ids": ["US1", "US2"], "conflict_type": "cross_cutting_concern", "description": "detailed conflict explanation"}},
      {{"story_ids": ["US1", "US5"], "conflict_type": "cross_cutting_concern", "description": "detailed conflict explanation"}},
    ]
  }},
  "unique": {{
    "has_duplicates": true,
    "violations": [
      {{
        "story_ids": ["US3", "US4"],
        "relationship_type": "purposeIsMeans",
        "details": {{
          "end_of_story": "US3",
          "means_of_story": "US4",
          "matched_text": "view landmark locations"
        }},
        "explanation": "US3's rationale (end) is US4's feature (means)",
        "recommendation": "make_explicit_dependency",
        "is_violation": true
      }},
      {{
        "story_ids": ["US5", "US6"],
        "relationship_type": "diffRoleSameStory",
        "details": {{
          "role_match": false,
          "roles": ["Manager", "Employee"],
          "action_match": true,
          "object_match": true
        }},
        "explanation": "Different roles requesting same functionality - acceptable pattern",
        "recommendation": "keep_separate",
        "is_violation": false
      }}
    ],
  }},
  "uniform": {{
    "is_uniform": true,
    "standard_template": "most common format detected",
    "deviations": [
      {{"story_id": "US1", "actual_format": "format used", "expected_format": "standard format", "reason": "explanation"}}
    ]
  }},
  "independent": {{
    "has_dependencies": true,
    "dependencies": [
      {{"story_id": "US1", "depends_on": ["US2"], "dependency_type": "causal", "description": "explanation", "is_avoidable": false}}
    ]
  }},
  "complete": {{
    "is_complete": false,
    "missing_stories": [
      {{"description": "missing story description", "reason": "why it's needed", "related_stories": ["US1", "US3"], "category": "CRUD/workflow/cross-cutting"}}
    ]
  }},
  "set_summary": {{
    "total_stories": 10,
    "conflicts_count": 3,
    "duplicates_count": 4,
    "non_uniform_count": 1,
    "dependencies_count": 2,
    "missing_stories_count": 3
  }}
}}
"""

In [ ]:

# ─────────────────────────────────────────────────────────────────────────
# API KEYS
# ─────────────────────────────────────────────────────────────────────────
DEEPINFRA_API_KEY  = "ADD YOUR API KEY"
DEEPINFRA_BASE_URL = "https://api.deepinfra.com/v1/openai/chat/completions"
ANTHROPIC_API_KEY  = "ADD YOUR API KEY"
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/messages"

MODEL_DEEPSEEK = "deepseek-ai/DeepSeek-V3.2"
MODEL_OPENAI   = "openai/gpt-oss-120b"
MODEL_SONNET   = "claude-sonnet-4-6"
MODEL_DEEPSEEKTER = "deepseek-ai/DeepSeek-V3.1-Terminus"
MODEL_OPENAI20   = "openai/gpt-oss-20b"


def call_model(model_key, system_prompt, user_prompt,
               temperature=0.3, max_tokens=20000):
    if model_key == "deepseek":
        return call_deepinfra(MODEL_DEEPSEEK, system_prompt, user_prompt,
                              temperature, max_tokens)
    elif model_key == "deepseekT":
        return call_deepinfra(MODEL_DEEPSEEKTER, system_prompt, user_prompt,
                              temperature, max_tokens)

    elif model_key == "openai":
        return call_deepinfra(MODEL_OPENAI, system_prompt, user_prompt,
                              temperature, max_tokens)
    elif model_key == "openai20":
        return call_deepinfra(MODEL_OPENAI20, system_prompt, user_prompt,
                              temperature, max_tokens)

    elif model_key == "sonnet":
        return call_anthropic(MODEL_SONNET, system_prompt, user_prompt,
                              temperature, max_tokens)
    else:
        raise ValueError(f"Unknown model key: {model_key}")

# ─────────────────────────────────────────────────────────────────────────
# ENSEMBLE RULES
# ─────────────────────────────────────────────────────────────────────────
ENSEMBLE_RULES = {
    # Individual criteria
    "well_formed":        ("OR",  ["openai"]),
    "full_sentence":      ("OR",  ["openai"]),
    "uniform":            ("OR",  ["openai"]),
    "complete":           ("OR",  ["openai"]),

    "atomic":             ("MAJ", ["openai", "deepseek"]),               # ≥2-of-2 (AND)
    "minimal":            ("OR", ["deepseekT", "openai20"]),  # OR-of-2
    "conceptually_sound": ("MAJ", ["openai", "deepseekT"]),              # ≥2-of-2 (AND)
    "problem_oriented":   ("OR",  ["openai", "openai20"]),     # OR-of-3
    "unambiguous":        ("OR",  ["deepseekT"]),                        # single model
    "estimatable":        ("OR",  ["deepseekT", "openai", "openai20"]),  # OR-of-3

    # Set criteria
    "conflict_free":      ("OR", ["openai", "sonnet", "deepseek"]),     # OR-of-3
    "unique":             ("OR",  ["sonnet", "deepseek", "openai"]),     # OR-of-3
    "independent":        ("OR", ["sonnet", "openai", "deepseek", "deepseekT"]),            # OR-of-2
}
INDIVIDUAL_CRITERIA = [
    "well_formed", "atomic", "minimal", "conceptually_sound",
    "problem_oriented", "unambiguous", "full_sentence", "estimatable"
]
SET_CRITERIA = ["conflict_free", "unique", "uniform", "independent",
                "complete", "problem_oriented"]  # problem_oriented included here for sonnet

# Model routing
# deepseek: individual + set
# openai:   individual only (including problem_oriented)
# sonnet:   set only (including problem_oriented — free ride on the set call)
MODEL_SCOPE = {
    "deepseek": ["set","individual"],
    "openai":   ["set","individual"],
    "sonnet":   ["set"],
    "openai20":   ["individual"],
    "deepseekT":   ["set","individual"],

}

# ─────────────────────────────────────────────────────────────────────────
# API CALLERS
# ─────────────────────────────────────────────────────────────────────────
def call_deepinfra(model_id, system_prompt, user_prompt,
                   temperature=0.3, max_tokens=15000, max_retries=3):
    headers = {"Authorization": f"Bearer {DEEPINFRA_API_KEY}",
               "Content-Type": "application/json"}
    payload = {
        "model": model_id,
        "messages": [{"role": "system", "content": system_prompt},
                     {"role": "user",   "content": user_prompt}],
        "temperature": temperature, "max_tokens": max_tokens,
        "response_format": {"type": "json_object"}
    }
    for attempt in range(max_retries):
        try:
            print(f"  🔄 DeepInfra/{model_id.split('/')[-1]} (attempt {attempt+1})...")
            r = requests.post(DEEPINFRA_BASE_URL, headers=headers,
                              json=payload, timeout=500)
            r.raise_for_status()
            content = r.json()["choices"][0]["message"]["content"]
            print(f"  ✓ {r.json().get('usage',{}).get('total_tokens','?')} tokens")
            return content
        except Exception as e:
            print(f"  ✗ Attempt {attempt+1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(5 ** attempt)
            else:
                raise

def call_anthropic(model_id, system_prompt, user_prompt,
                   temperature=0.3, max_tokens=15000, max_retries=3):
    headers = {"x-api-key": ANTHROPIC_API_KEY,
               "anthropic-version": "2023-06-01",
               "Content-Type": "application/json"}
    payload = {"model": model_id, "system": system_prompt,
               "messages": [{"role": "user", "content": user_prompt}],
               "temperature": temperature, "max_tokens": max_tokens}
    for attempt in range(max_retries):
        try:
            print(f"  🔄 Anthropic/{model_id} (attempt {attempt+1})...")
            r = requests.post(ANTHROPIC_BASE_URL, headers=headers,
                              json=payload, timeout=300)
            r.raise_for_status()
            content = r.json()['content'][0]['text']
            #content = parse_json_response(content)
            print("  ✓ done")
            return content
        except Exception as e:
            print(f"  ✗ Attempt {attempt+1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise

def parse_json_response(content):
    if not content or not isinstance(content, str):
        return {"error": "empty content"}
    for fn in [
        lambda c: json.loads(c),
        lambda c: json.loads(re.sub(r'```json\s*|\s*```', '', c)),
        lambda c: json.loads(c[c.find('{'):c.rfind('}')+1]),
    ]:
        try:
            return fn(content)
        except Exception:
            continue
    return {"error": "JSON parse failed", "raw": content[:500]}

print("✓ API config loaded. Calls per group:")
print("  deepseek : 1 per story (individual) + 1 for set")
print("  openai   : 1 per story (individual, incl. problem_oriented)")
print("  sonnet   : 1 for set (incl. problem_oriented — no extra call)")


In [ ]:
def parse_json_response(content, strict=False):
    """
    Parse JSON from LLM response with robust error handling

    Args:
        content: Raw response content from LLM
        strict: If True, raise exception on parse failure. If False, return error dict

    Returns:
        dict: Parsed JSON object or error dictionary
    """
    if not content or not isinstance(content, str):
        error_msg = "Empty or invalid content"
        print(f"✗ {error_msg}")
        if strict:
            raise ValueError(error_msg)
        return {"error": error_msg, "raw_content": content}

    try:
        # Method 1: Direct parse (for clean JSON)
        parsed = json.loads(content)
        print("✓ JSON parsed successfully (direct)")
        return parsed

    except json.JSONDecodeError:
        # Method 2: Extract JSON from markdown code blocks
        try:
            # Remove markdown JSON blocks
            content_clean = re.sub(r'```json\s*|\s*```', '', content)
            parsed = json.loads(content_clean)
            print("✓ JSON parsed successfully (removed markdown)")
            return parsed

        except json.JSONDecodeError:
            # Method 3: Find JSON object in text
            try:
                # Find first { to last }
                start_idx = content.find('{')
                end_idx = content.rfind('}') + 1

                if start_idx != -1 and end_idx > start_idx:
                    json_str = content[start_idx:end_idx]

                    # Try to fix common issues
                    # Remove trailing commas
                    json_str = re.sub(r',\s*}', '}', json_str)
                    json_str = re.sub(r',\s*]', ']', json_str)

                    parsed = json.loads(json_str)
                    print("✓ JSON parsed successfully (extracted and cleaned)")
                    return parsed
                else:
                    raise ValueError("No JSON object found in content")

            except (json.JSONDecodeError, ValueError) as e:
                error_msg = f"Failed to parse JSON: {str(e)}"
                print(f"✗ {error_msg}")
                print(f"{content}")

                if strict:
                    raise ValueError(error_msg)

                return {
                    "error": error_msg,
                    "parse_attempts": ["direct", "markdown_removed", "extracted"],
                    "raw_content": content  # First 1000 chars for debugging
                }



In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from google.colab import files


# ─────────────────────────────────────────────────────────────────────────
# UPLOAD CSV
# ─────────────────────────────────────────────────────────────────────────
# print("Upload your CSV file...")
# uploaded = files.upload()
csv_fname = '/content/A1-20a27.csv' #[k for k in uploaded if k.endswith('.csv')][0]
df_raw = pd.read_csv(csv_fname)
print(f"✓ Loaded {len(df_raw)} rows")
print(f"  Abstracts : {sorted(df_raw['ID'].unique())}")
print(f"  LLMs      : {sorted(df_raw['LLM'].unique())}")
df_raw.head(3)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONFIDENCE CALCULATION
# ─────────────────────────────────────────────────────────────────────────
def compute_ensemble(criterion, model_votes):
    """
    Apply ensemble rule. Returns (label, confidence, rule_used, votes_used).

    Confidence:
      prior       : 1.0
      1vote       : 1.0
      OR  both=1  : 1.0 | one fires : 0.5 | both=0 : 1.0
      AND both=1  : 1.0 | split     : 0.5 | both=0 : 1.0
    """
    rule, models = ENSEMBLE_RULES[criterion]

    # if rule == "prior0": return 0, 1.0, "prior0", {}
    # if rule == "prior1": return 1, 1.0, "prior1", {}

    votes = {m: model_votes.get(m) for m in models
             if model_votes.get(m) is not None}
    if not votes:
        return -1, 0.0, rule, {}

    if rule == "1vote":
        m = models[0]
        v = votes.get(m)
        if v is None: return -1, 0.0, "1vote", {}
        return int(v), 1.0, "1vote", {m: v}

    scores  = list(votes.values())
    n       = len(scores)
    n_fire  = sum(scores)

    if rule == "OR":
        label = 1 if n_fire > 0 else 0
        conf  = 1.0 if (n_fire == n or n_fire == 0) else round(n_fire / n, 2)
        return label, conf, "OR", votes

    if rule == "MAJ":
        label = 1 if n_fire > 1 else 0
        conf  = 1.0 if (n_fire == n or n_fire == 0) else round(n_fire / n, 2)
        return label, conf, "OR", votes


    if rule == "AND":
        label = 1 if n_fire == n else 0
        conf  = 1.0 if (n_fire == n or n_fire == 0) else round((n - n_fire) / n, 2)
        return label, conf, "AND", votes

    return -1, 0.0, rule, votes


# ─────────────────────────────────────────────────────────────────────────
# PHASE 1a: Individual criteria  (well_formed → estimatable)
# ─────────────────────────────────────────────────────────────────────────
def evaluate_individual(stories):
    """One call per model using QUS_INDIVIDUAL_EVALUATION_PROMPT."""
    stories_text = "\n".join([f"US{i+1}: {s}" for i, s in enumerate(stories)])
    max_tok      = len(stories) * 1500   # ~1000 tokens/story for individual
    results      = {}

    for model in ["openai", "deepseekT","openai20", "deepseek"]:
        prompt = QUS_INDIVIDUAL_EVALUATION_PROMPT.format(user_stories_list=stories_text)
        try:
            raw    = call_model(model, QUS_EVALUATION_SYSTEM_PROMPT, prompt, max_tokens=max_tok)
            parsed = parse_json_response(raw)
            if not parsed:
                raise ValueError("Empty parse result")
            results[model] = parsed
            print(f"  ✓ [{model}] individual done")
        except Exception as e:
            print(f"  ⚠ [{model}] individual failed: {e}")
            results[model] = {}

    return results   # {model: {criterion: {USn: {score, reason}}}}


# ─────────────────────────────────────────────────────────────────────────
# PHASE 1b: Set-level criteria  (conflict_free → complete)
# ─────────────────────────────────────────────────────────────────────────
def evaluate_set(stories):
    """One call per model using QUS_SET_EVALUATION_PROMPT."""
    stories_text = "\n".join([f"US{i+1}: {s}" for i, s in enumerate(stories)])
    max_tok      = len(stories) * 1500    # set criteria are more compact
    results      = {}

    for model in ["openai", "deepseek","deepseekT", "sonnet"]:
        prompt = QUS_SET_EVALUATION_PROMPT.format(user_stories_list=stories_text)
        try:
            raw    = call_model(model, QUS_EVALUATION_SYSTEM_PROMPT, prompt, max_tokens=max_tok)
            parsed = parse_json_response(raw)
            print(parsed)
            if not parsed:
                raise ValueError("Empty parse result")
            results[model] = parsed
            print(f"  ✓ [{model}] set done")
        except Exception as e:
            print(f"  ⚠ [{model}] set failed: {e}")
            results[model] = {}

    return results   # {model: {criterion: {...}}}


# ─────────────────────────────────────────────────────────────────────────
# EXTRACT SCORES — routes each criterion to the right source dict
# ─────────────────────────────────────────────────────────────────────────
INDIVIDUAL_CRITERIA = {
    "well_formed", "atomic", "minimal", "conceptually_sound",
    "problem_oriented", "unambiguous", "full_sentence", "estimatable",
}
SET_CRITERIA = {
    "conflict_free", "unique", "uniform", "independent", "complete",
}

def extract_scores(eval_data, story_id, kind):
    def make(score, reason=""):
        return {"score": int(score), "reason": reason}

    def from_per_story(block):
        entry = block.get(story_id)
        if entry is None:
            return None
        if isinstance(entry, dict):
            return make(entry.get("score", 1), entry.get("reason", ""))
        return make(int(entry))

    scores = {}

    if kind == "individual":
        for criterion in INDIVIDUAL_CRITERIA:
            result = from_per_story(eval_data.get(criterion, {}))
            if result is not None:
                scores[criterion] = result

    elif kind == "set":

        # ── conflict_free ────────────────────────────────────────────────
        cf       = eval_data.get("conflict_free", {})
        my_conflicts = [
            c for c in cf.get("conflicts", [])
            if story_id in c.get("story_ids", [])
        ]
        scores["conflict_free"] = make(
            score  = 1 if my_conflicts else 0,
            reason = "; ".join(
                f"Conflicts with {[s for s in c['story_ids'] if s != story_id]}: "
                f"{c.get('description', c.get('conflict_type', ''))}"
                for c in my_conflicts
            )
        )

        # ── unique ───────────────────────────────────────────────────────
        uq = eval_data.get("unique", {})
        my_violations = [
            v for v in uq.get("violations", [])
            if v.get("is_violation", True) and story_id in v.get("story_ids", [])
        ]
        scores["unique"] = make(
            score  = 1 if my_violations else 0,
            reason = "; ".join(
                f"{v.get('relationship_type', '')} with "
                f"{[s for s in v['story_ids'] if s != story_id]}: "
                f"{v.get('explanation', '')}"
                for v in my_violations
            )
        )

        # ── independent ──────────────────────────────────────────────────
        ind = eval_data.get("independent", {})
        my_deps = [
            d for d in ind.get("dependencies", [])
            if d.get("story_id") == story_id
        ]
        scores["independent"] = make(
            score  = 1 if my_deps else 0,
            reason = "; ".join(
                f"Depends on {d.get('depends_on', [])}"
                f" ({d.get('dependency_type', '')}): {d.get('description', '')}"
                for d in my_deps
            )
        )

        # ── uniform ──────────────────────────────────────────────────────
        uf = eval_data.get("uniform", {})
        my_deviations = [
            d for d in uf.get("deviations", [])
            if d.get("story_id") == story_id
        ]
        scores["uniform"] = make(
            score  = 1 if my_deviations else 0,
            reason = "; ".join(
                f"Expected '{d.get('expected_format', '')}', "
                f"got '{d.get('actual_format', '')}': {d.get('reason', '')}"
                for d in my_deviations
            )
        )

        # ── complete ─────────────────────────────────────────────────────
        # complete is set-level (not per-story), so reason lists ALL missing stories
        comp         = eval_data.get("complete", {})
        print(comp)
        is_complete  = comp.get("is_complete", True)
        missing      = comp.get("missing_stories", [])
        # surface only missing stories that are related to this story
        my_missing = [
            m for m in missing
            if story_id in m.get("related_stories", [])
        ]
        scores["complete"] = make(
            score  = 0 if is_complete else 1,
            reason = "; ".join(
                f"[{m.get('category', '')}] {m.get('description', '')}: {m.get('reason', '')}"
                for m in my_missing
            )
        )

    print(scores)
    return scores

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# RUN FULL EXPERIMENT  —  with per-group checkpointing
# ─────────────────────────────────────────────────────────────────────────

import json, os
import pandas as pd

RAW_SCORES_PATH   = "raw_model_scores.json"
ENSEMBLE_OUT_PATH = "ensemble_labels.csv"


# ── Checkpoint helpers ────────────────────────────────────────────────────
def load_checkpoint():
    """Load existing raw scores from disk (empty dict if none)."""
    if os.path.exists(RAW_SCORES_PATH):
        with open(RAW_SCORES_PATH) as f:
            data = json.load(f)
        print(f" Resuming — {len(data)} group(s) already completed: {list(data.keys())}")
        return data
    return {}


def save_checkpoint(raw):
    """Overwrite checkpoint file with current state."""
    with open(RAW_SCORES_PATH, "w") as f:
        json.dump(raw, f, indent=2)


def collect_raw_scores(groups, raw=None):
    """
    Calls all models for each group and extracts per-story criterion scores.
    Saves after every group so runs can be interrupted and resumed.
    Skips any group_key already present in `raw`.

    Returns:
        raw[group_key] = {
            "meta":   { abstract_id, llm_name, stories, orig_rows },
            "scores": { story_id: { model: { criterion: {score, reason} } } }
        }
    """
    if raw is None:
        raw = {}

    pending = [
        ((abstract_id, llm_name), group)
        for (abstract_id, llm_name), group in groups
        if f"{abstract_id}_{llm_name}" not in raw
    ]
    print(f"{len(groups) - len(pending)} group(s) skipped (already done), "
          f"{len(pending)} remaining.\n")

    for (abstract_id, llm_name), group in pending:
        gk      = f"{abstract_id}_{llm_name}"
        stories = group["User story"].tolist()
        print(f"\n[{gk}] {len(stories)} stories — calling models...")

        try:
            print("  → Individual criteria...")
            indiv_results = evaluate_individual(stories)   # ✅ added

            print("  → Set-level criteria...")
            set_results = evaluate_set(stories)
        except Exception as e:
            print(f"  ✗ {gk} failed: {e}")
            continue

        group_scores = {}
        for i in range(len(stories)):
            story_id     = f"US{i+1}"
            story_scores = {}

            for model in ["openai", "deepseek", "deepseekT"]:
                indiv = extract_scores(indiv_results.get(model, {}), story_id, "individual")  # ✅ kind param
                sets  = extract_scores(set_results.get(model,   {}), story_id, "set")         # ✅ kind param
                story_scores[model] = {**indiv, **sets}    # ✅ merge both
            for model in ["openai20"]:
                indiv = extract_scores(indiv_results.get(model, {}), story_id, "individual")  # ✅ kind param
                # sets  = extract_scores(set_results.get(model,   {}), story_id, "set")         # ✅ kind param
                story_scores[model] = {**indiv}    # ✅ merge both
            for model in ["sonnet"]:
                # indiv = extract_scores(indiv_results.get(model, {}), story_id, "individual")  # ✅ kind param
                sets  = extract_scores(set_results.get(model,   {}), story_id, "set")         # ✅ kind param
                story_scores[model] = {**sets}    # ✅ merge both

            group_scores[story_id] = story_scores

        raw[gk] = {
            "meta": {
                "abstract_id": abstract_id,
                "llm_name":    llm_name,
                "stories":     stories,
                "orig_rows": {
                    f"US{i+1}": {"title": orig.get("title", ""), "ID": orig.get("ID", "")}
                    for i, (_, orig) in enumerate(group.iterrows())
                },
            },
            "scores": group_scores,
        }

        save_checkpoint(raw)
        print(f"  ✓ [{gk}] saved ({len(group_scores)} stories)")

    print(f"\n✓ Phase 1 complete — {len(raw)} groups in {RAW_SCORES_PATH}")
    return raw

# ── Phase 2: Ensemble rules ───────────────────────────────────────────────
def apply_ensemble_rules(raw):
    output_rows = []

    for gk, group_data in raw.items():
        meta   = group_data["meta"]
        scores = group_data["scores"]

        for story_id, model_scores in scores.items():
            story_idx  = int(story_id[2:]) - 1
            story_text = meta["stories"][story_idx]
            orig       = meta["orig_rows"].get(story_id, {})

            row = {
                "group":       gk,
                "story_id":    story_id,
                "story_text":  story_text,
                "abstract_id": meta["abstract_id"],
                "llm":         meta["llm_name"],
                "title":       orig.get("title", ""),
                "ID":          orig.get("ID", ""),
            }

            for criterion in ENSEMBLE_RULES:
                model_votes   = {}
                model_reasons = {}

                for model, criterion_scores in model_scores.items():
                    entry = criterion_scores.get(criterion)
                    if entry is not None:
                        # ✅ entry is {"score": int, "reason": str} — unpack both
                        model_votes[model]   = entry["score"]    # int → compute_ensemble
                        model_reasons[model] = entry.get("reason", "")

                label, conf, rule_used, votes = compute_ensemble(criterion, model_votes)

                row[f"{criterion}_label"]   = label
                row[f"{criterion}_conf"]    = conf
                row[f"{criterion}_rule"]    = rule_used
                row[f"{criterion}_votes"]   = json.dumps(votes)
                row[f"{criterion}_reasons"] = json.dumps(model_reasons)

            output_rows.append(row)

    return pd.DataFrame(output_rows)


# ── Orchestration ─────────────────────────────────────────────────────────
groups = list(df_raw.groupby(["Abstract", "LLM"]))
print(f"Total groups: {len(groups)}\n")

raw_scores = load_checkpoint()              # load whatever was saved before
raw_scores = collect_raw_scores(groups, raw_scores)   # skip completed, run rest

# Phase 2 — always re-derived from the full raw scores
results_df = apply_ensemble_rules(raw_scores)
results_df.to_csv(ENSEMBLE_OUT_PATH, index=False)

print(f"\n✓ {len(results_df)} stories written → {ENSEMBLE_OUT_PATH}")
results_df.head(3)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# SUMMARY REPORT  — no ground truth required
# Shows: defect rate, confidence distribution, low-confidence flags
# ─────────────────────────────────────────────────────────────────────────
CRITERIA_ALL = list(ENSEMBLE_RULES.keys())
NON_TRIVIAL  = [c for c in CRITERIA_ALL
                if ENSEMBLE_RULES[c][0] not in ("prior0","prior1")]

print("=" * 80)
print(f"{'Criterion':<22} {'Rule':<8} {'Models':<24} "
      f"{'Defect%':>8} {'AvgConf':>8} {'LowConf%':>9}")
print("-" * 80)

for crit in CRITERIA_ALL:
    rule, models = ENSEMBLE_RULES[crit]
    lc = f"{crit}_label"
    cc = f"{crit}_conf"
    if lc not in results_df.columns:
        continue
    labels = results_df[lc].replace(-1, np.nan).dropna()
    confs  = results_df[cc].dropna()

    defect_pct  = labels.mean() * 100   if len(labels) else 0
    avg_conf    = confs.mean()          if len(confs)  else 0
    low_conf_pct= (confs < 0.75).mean() * 100 if len(confs) else 0

    ms = "+".join(models) if models else "—"
    print(f"{crit:<22} {rule:<8} {ms:<24} "
          f"{defect_pct:>7.1f}% {avg_conf:>8.3f} {low_conf_pct:>8.1f}%")

print("=" * 80)
print()

# ─────────────────────────────────────────────────────────────────────────
# CONFIDENCE DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────
print("Confidence Distribution by Criterion")
print("=" * 68)
print(f"{'Criterion':<22} {'HIGH ≥0.9':>11} {'MED 0.5–0.9':>13} {'LOW <0.5':>10}")
print("-" * 68)
for crit in NON_TRIVIAL:
    cc = f"{crit}_conf"
    if cc not in results_df.columns: continue
    c = results_df[cc].dropna()
    if not len(c): continue
    hi  = (c >= 0.90).mean()*100
    med = ((c >= 0.50) & (c < 0.90)).mean()*100
    lo  = (c <  0.50).mean()*100
    print(f"{crit:<22} {hi:>10.1f}% {med:>12.1f}% {lo:>9.1f}%")
print("=" * 68)

# ─────────────────────────────────────────────────────────────────────────
# DEFECT COUNT PER STORY (overall quality score)
# ─────────────────────────────────────────────────────────────────────────
print()
label_cols = [f"{c}_label" for c in NON_TRIVIAL if f"{c}_label" in results_df.columns]
results_df["n_defects"]  = results_df[label_cols].replace(-1, 0).sum(axis=1)
results_df["pct_defects"]= results_df["n_defects"] / len(label_cols) * 100
conf_cols = [f"{c}_conf" for c in NON_TRIVIAL if f"{c}_conf" in results_df.columns]
results_df["avg_conf"]   = results_df[conf_cols].mean(axis=1)

print("Top 10 stories with most defects:")
top = results_df.nlargest(10, "n_defects")[
    ["group","story_id","story_text","n_defects","pct_defects","avg_conf"]
]
pd.set_option('display.max_colwidth', 80)
print(top.to_string(index=False))


---
## QUASE-Enhance

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MULTI-AGENT ENHANCEMENT — SETUP
# ═══════════════════════════════════════════════════════════════════════════
#
# Design principles
# ─────────────────
# • The same model that fired a defect is responsible for fixing it
#     1vote  → that model
#     OR/MAJ → highest-priority model that voted 1 (sonnet > openai > deepseek)
#     AND    → highest-priority model that voted 1
#
# • conflict_free and unambiguous require human input before rewriting:
#     responsible model generates ONE focused question → user answers →
#     responsible model rewrites using that answer
#
# • Rewriting is sequential per story: criterion 1 output feeds criterion 2
#   (atomic first so a split becomes the base for all downstream fixes)
#
# • After each rewrite the responsible model does a quick self-check
# ═══════════════════════════════════════════════════════════════════════════

from collections import defaultdict, OrderedDict

# ── Config ──────────────────────────────────────────────────────────────────
DEFECTS_TO_FIX     = {"DEFECT_HIGH_CONF", "DEFECT_MED_CONF", "REVIEW_SPLIT"}
HUMAN_INPUT_NEEDED = {"conflict_free", "unambiguous"}
MAX_ROUNDS         = 2    # rewrite passes per criterion before giving up

# Process criteria in this order so atomic splits happen first
CRITERIA_PRIORITY = [
    "atomic", "minimal", "conceptually_sound", "problem_oriented",
    "unambiguous", "estimatable", "conflict_free", "unique", "independent",
    "complete", "uniform",
]

# ── Helper: always return a dict from call_model ─────────────────────────────
def safe_parse(raw):
    """Handle both dict (sonnet) and string (deepseek/openai) returns."""
    if isinstance(raw, dict):
        return raw
    return parse_json_response(raw)


# ── Responsible-model routing ─────────────────────────────────────────────────
def get_responsible_model(criterion, votes_json):
    """
    Return the model key that should fix this defect.
    Always picks the model that actually voted 1.
    If multiple models fired: sonnet > openai > deepseek.
    """
    rule, models = ENSEMBLE_RULES.get(criterion, ("1vote", ["deepseek"]))

    if rule in ("prior0", "prior1"):
        return None  # trivial — nothing to fix

    try:
        votes = json.loads(votes_json) if isinstance(votes_json, str) else (votes_json or {})
    except Exception:
        votes = {}

    fired = [m for m, v in votes.items() if int(v) == 1]

    for preferred in ["sonnet", "openai", "deepseek", "openai20", "deepseekT"]:
        if preferred in fired:
            return preferred

    return models[0] if models else "deepseek"


# ── Single source of truth: full criterion definitions ────────────────────────
# Feeds ONLY the agent prompts (question / rewrite / reeval).
# The evaluation prompts (QUS_INDIVIDUAL_EVALUATION_PROMPT,
# QUS_SET_EVALUATION_PROMPT) are defined in Cell 2 and are NOT touched here.

CRITERION_DETAIL = {

    "well_formed": """\
### Well-formed
**Definition:** A user story includes at least a role and a means.
**Standard template:** "As a <role>, I want <means>, [so that <end>]"

**Evaluation:**
- Does it include a role (who)?
- Does it include a means (what)?
- Score 0 if BOTH present, 1 if either missing""",

    "atomic": """\
### Atomic
**Definition:** A user story expresses a requirement for exactly one feature.

**DETECTION STRATEGY - Follow these steps:**

**STEP 1: Scan the MEANS for conjunctions**
- Search for: "and", "or", "&", "/"
- Flag each occurrence for review

**STEP 2: For each conjunction found, check BOTH sides:**
- Left side: Does it describe a distinct feature/action/component?
- Right side: Does it describe a distinct feature/action/component?
- If BOTH sides are features/actions/components → VIOLATION

**STEP 3: Common violation patterns:**
- x "AI **and** real-time data" (two data sources)
- x "sensors **and** algorithms" (two components)
- x "optimize **and** reduce" (two separate goals)
- x "login **or** register" (two distinct operations)

**Domain-aware exceptions (NOT violations):**
- ✓ "bread and butter" (idiom, single concept)
- ✓ "terms and conditions" (inseparable legal pair)
- ✓ "Research and Development" (single department name)

**Compound nouns / Adjective pairs (NOT violations):**
- ✓ "peanut butter and jelly sandwich" (single item)
- ✓ "fast and efficient" (adjectives modifying same feature)""",

    "minimal": """\
### Minimal
**Definition:** A user story contains nothing more than role, means, and ends.

**Violations:**
- Implementation hints, technical details beyond feature description
- UI mockup references, design specifications
- Testing instructions, notes in parentheses
- Comments that belong in description/acceptance criteria
- Multiple sentences with extra context

**Evaluation:**
- Does it contain ONLY role, means, and optional ends?
- Is there text in parentheses, brackets, or after dashes/semicolons?
- Are there implementation details that should be in comments?""",

    "conceptually_sound": """\
### Conceptually sound
**Definition:** The means expresses a CONCRETE FEATURE and the ends expresses a RATIONALE (not anything else).

**PART 1: Is the MEANS a concrete feature?**

Concrete Features (✓ Valid means):
- Specific functionality: "navigation system", "payment processing"
- Observable capability: "detect obstacles", "send notifications"
- Deliverable output: "generate report", "display dashboard"

NOT Features (✗ Invalid means):
- x Vague exploration: "explore", "investigate", "research"
- x Meta-activities: "analyze options", "evaluate alternatives"
- x Placeholder intent: "look into", "figure out", "work on"

Detection: Ask "Can this be IMPLEMENTED and TESTED in a sprint?"

**PART 2: Does the ENDS express rationale (WHY)?**

Valid Ends (✓ Rationale): user benefit, business value, motivation
Invalid Ends (✗ NOT rationale):
- x Hidden feature: "so that I can see landmark locations" (= another story)
- x Implementation detail: "so that the system uses caching" (HOW, not WHY)
- x Technical explanation: "so that the database stays synchronized"

**Scoring:**
- Score 1 if means is exploration/meta-activity
- Score 1 if ends describes another feature or implementation detail
- Score 0 only if means = concrete feature AND ends (if present) = rationale""",

    "problem_oriented": """\
### Problem-oriented
**Definition:** A user story only specifies the problem (WHAT is needed), not the solution (HOW to build it).

**Evaluation:**
- Describes WHAT feature is needed? → Problem (Score 0)
- Prescribes HOW to build it / names specific technology? → Solution (Score 1)
- General technology mention only, no details → Problem (Score 0)
- Specific tech/brand/algorithm/UI placement mentioned → Score 1""",

    "unambiguous": """\
### Unambiguous
**Definition:** A user story avoids terms or abstractions that lead to multiple interpretations.

**Violations:**
- Vague terms: "content", "data", "information", "stuff", "things"
- Ambiguous pronouns: "it", "this", "that" without clear referent
- Abstract concepts without measurable criteria: "efficient", "fast", "good"
- Superclass terms: "vehicle" when you mean "car"
- Context-dependent words: "soon", "large", "many" without quantification
- Unclear scope: "manage users" (create? edit? delete? all?)

**Acceptable:**
- Specific terms: "video", "user profile", "payment transaction"
- Domain terminology: "autonomous vehicle", "navigation system"
- Quantified when possible: "within 2 seconds", "at least 10 items" """,

    "full_sentence": """\
### Full sentence
**Definition:** A user story is a well-formed, grammatically correct full sentence.

**Violations:**
- Sentence fragments: "Login functionality", "User profile update"
- Missing punctuation, grammatical errors
- Not following standard sentence structure

**Evaluation:**
- Is it a grammatically correct, complete sentence?
- Does it have proper capitalization and punctuation?""",

    "estimatable": """\
### Estimatable
**Definition:** A story does not denote a coarse-grained requirement that is difficult to plan and prioritize.

**Violations:**
- Too large/vague to estimate effort accurately
- Unclear scope — could be 1 day or 1 month of work
- Epic-sized story not broken down
- Exploratory stories: "explore options", "investigate solutions"
- Multiple features bundled together

**Acceptable:**
- Clear, bounded scope; single feature with understood complexity
- Enough detail to estimate effort; completable in one sprint (1–3 weeks)""",

    "conflict_free": """\
### Conflict-free
**Definition:** A user story should not be inconsistent with any other user story.

**Violations:**
- Contradictory business rules across stories
- Inconsistent access control or permissions
- Direct contradiction, goal conflicts, resource conflicts

**Cross-Cutting Concern patterns:**
- Data collection (needs data) vs Privacy protection (minimize data)
- System autonomy (AI decides) vs Security controls (human oversight)
- Performance optimization (speed) vs Safety regulations (compliance checks)
- Functionality (features) vs Regulatory compliance (restrictions)

**Evaluation:** Identify pairs of stories that contradict each other.""",

    "unique": """\
### Unique
**Definition:** Every user story is unique; duplicates are avoided.

**Duplicate types to detect:**
- TYPE 1 Full Duplicate: identical text (case-insensitive)
- TYPE 2 Semantic Duplicate: same role + action + object, different wording
- TYPE 3 Different Means, Same End: "overlapping ends"
- TYPE 4 Same Means, Different End: "same feature, different rationale"
- TYPE 5 Different Role, Same Means/End: usually ACCEPTABLE ("diffRoleSameStory")
- TYPE 6 End = Means (purposeIsMeans): story λ1's end matches another story's means
  → λ1's rationale is actually another story's feature; violates uniqueness AND independence""",

    "uniform": """\
### Uniform
**Definition:** All user stories in a specification employ the same template.

**Check for consistency in:**
- Role format: "As a" vs "As an"
- Means format: "I want" vs "I want to" vs "I am able to" vs "I can" vs "I need"
- Ends format: "so that" vs "in order to" vs "so" vs no ends marker
- Punctuation: commas after role, periods at end

**Violations:**
- Mixing different means indicators inconsistently
- Some stories have ends, others don't
- Inconsistent punctuation patterns""",

    "independent": """\
### Independent
**Definition:** The user story is self-contained with no inherent dependencies on other stories.

**Dependency Types:**
- Type 1 Causal (CRUD chains): "Create X" must happen before "Edit X" or "Delete X"
- Type 2 Functional: Story A requires Story B's functionality to work
- Type 3 Data: Story references data that another story creates
- Type 4 Is-A (Superclass/Subclass): story about "content" depends on stories defining content types

**Evaluation:** Identify, classify, and flag whether each dependency is avoidable.""",

    "complete": """\
### Complete
**Definition:** Implementing the set creates a feature-complete application with no missing steps.

**Check for:**
- Missing CRUD operations (can Create but no Read/View story, etc.)
- Missing user workflows (story for outcome but not for setup/prerequisites)
- Missing cross-cutting concerns (no error handling, no authentication)
- Referenced but undefined features ("user profile" mentioned but no story creates it)""",
}


# ── Agent prompt helpers ──────────────────────────────────────────────────────
def _criterion_context(criterion):
    """Returns the full definition block for a criterion."""
    return CRITERION_DETAIL.get(criterion, f"Criterion: {criterion}")


def build_question_prompt(criterion, story, group_stories):
    other  = "\n".join(f"  • {s}" for s in group_stories if s != story)
    detail = _criterion_context(criterion)

    if criterion == "conflict_free":
        return f"""A user story conflicts with other stories in the backlog.

Conflicting story:
  "{story}"

Other stories in the backlog:
{other}

CRITERION DEFINITION:
{detail}

Your task: Identify the specific conflict and write ONE precise question for
the product owner that asks them to clarify priority or resolution strategy.
The question must be answerable and actionable — its answer should let you
rewrite the story to eliminate the conflict.

Return JSON: {{"question": "...", "conflict_summary": "brief description of the conflict"}}"""

    if criterion == "unambiguous":
        return f"""A user story contains ambiguous terms that could be interpreted multiple ways.

Story:
  "{story}"

Other stories for context:
{other}

CRITERION DEFINITION:
{detail}

Your task: Identify the specific ambiguous term(s) and write ONE precise
question for the product owner that asks them to clarify exactly what is meant.
The question must be answerable and actionable — its answer should let you
replace the vague term with a specific one.

Return JSON: {{"question": "...", "ambiguous_terms": ["term1", "term2"]}}"""

    return f"""A user story has a quality issue.

Story:
  "{story}"

Other stories for context:
{other}

CRITERION DEFINITION:
{detail}

Write ONE focused question to ask the product owner before rewriting.
Return JSON: {{"question": "..."}}"""


def build_rewrite_prompt(criterion, story, group_stories, user_answer=""):
    other        = "\n".join(f"  • {s}" for s in group_stories if s != story)
    answer_block = f"\nProduct owner clarification: {user_answer}" if user_answer else ""
    detail       = _criterion_context(criterion)

    base = f"""Rewrite the following user story to fix a quality violation.

Original story:
  "{story}"

CRITERION VIOLATED: {criterion}
{detail}{answer_block}

Other stories in the backlog (for context and consistency):
{other if other else "  (none)"}

Rules:
  • Keep the "As a <role>, I want <goal>, so that <benefit>" format
  • Preserve the original intent — do not change what is being requested
  • Fix ONLY the {criterion} violation
  • If the story must be split (atomic violation), return multiple stories"""

    if criterion == "atomic":
        return base + '\n\nReturn JSON: {"rewritten": ["story 1", "story 2"]}  ← split if needed, else one story'
    if criterion == "unique":
        return base + '\n\nReturn JSON: {"rewritten": ["distinct story"]} OR {"rewritten": []} to remove as true duplicate'
    return base + '\n\nReturn JSON: {"rewritten": ["improved story"]}'


def build_reeval_prompt(criterion, story, group_stories):
    other  = "\n".join(f"  • {s}" for s in group_stories if s != story)
    detail = _criterion_context(criterion)

    return f"""Does the following user story pass the {criterion} criterion?

Story:
  "{story}"

CRITERION DEFINITION:
{detail}

Other stories for context:
{other[:800] if other else "(none)"}

Apply the detection strategy above rigorously.
Answer with only: {{"passes": true}} or {{"passes": false, "reason": "brief"}}"""


# ── Core agent functions ──────────────────────────────────────────────────────
def agent_generate_question(criterion, story, group_stories, model_key):
    prompt = build_question_prompt(criterion, story, group_stories)
    try:
        raw  = call_model(model_key, QUS_EVALUATION_SYSTEM_PROMPT, prompt,
                          temperature=0.3, max_tokens=400)
        data = safe_parse(raw)
        return data.get("question", f"Please clarify the {criterion} issue in this story.")
    except Exception as e:
        return f"Please clarify the {criterion} issue in this story. (Error: {e})"


def agent_rewrite(criterion, story, group_stories, model_key, user_answer=""):
    prompt = build_rewrite_prompt(criterion, story, group_stories, user_answer)
    try:
        raw      = call_model(model_key, QUS_EVALUATION_SYSTEM_PROMPT, prompt,
                              temperature=0.4, max_tokens=800)
        data     = safe_parse(raw)
        rewrites = data.get("rewritten", [story])
        if not isinstance(rewrites, list):
            rewrites = [str(rewrites)]
        rewrites = [r.strip() for r in rewrites if r and r.strip()]
        return rewrites if rewrites else [story]
    except Exception as e:
        print(f"    ⚠ Rewrite failed: {e}")
        return [story]


def agent_reeval(criterion, story, group_stories, model_key):
    prompt = build_reeval_prompt(criterion, story, group_stories)
    try:
        raw  = call_model(model_key, QUS_EVALUATION_SYSTEM_PROMPT, prompt,
                          temperature=0.1, max_tokens=100)
        data = safe_parse(raw)
        return bool(data.get("passes", True))
    except Exception:
        return True   # assume fixed if re-eval fails


print("✓ Enhancement setup loaded")
print(f"  Criteria requiring human input : {HUMAN_INPUT_NEEDED}")
print(f"  Defect flags that trigger fix  : {DEFECTS_TO_FIX}")
print(f"  Max rewrite rounds per story   : {MAX_ROUNDS}")
print()
print("Responsible model routing:")
for crit, (rule, models) in ENSEMBLE_RULES.items():
    if rule in ("prior0", "prior1"): continue
    ex_votes = json.dumps({m: 1 for m in models})
    resp = get_responsible_model(crit, ex_votes)
    print(f"  {crit:<22} {rule:<6} → {resp}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 1 — COLLECT HUMAN CLARIFICATIONS
# ═══════════════════════════════════════════════════════════════════════════
# For every defect in conflict_free or unambiguous:
#   1. The responsible model generates a focused question
#   2. You answer it
#   3. Answers are stored in user_qa for Phase 2 (rewriting)
#
# All other defects will be rewritten automatically in Phase 2.
# ═══════════════════════════════════════════════════════════════════════════

# ── Build defect work list from flags_df ──────────────────────────────────
# Group all defects by story, sorted by criterion priority
work_list = defaultdict(list)   # (abstract_id, llm, story_id) → [criterion, ...]

for _, frow in flags_df[flags_df["flag"].isin(DEFECTS_TO_FIX)].iterrows():
    key = (frow["abstract_id"], frow["llm"], frow["story_id"])
    work_list[key].append({
        "criterion":  frow["criterion"],
        "confidence": frow["confidence"],
        "votes":      frow["votes"],
        "flag":       frow["flag"],
    })

# Sort each story's defects by CRITERIA_PRIORITY
for key in work_list:
    work_list[key].sort(
        key=lambda d: CRITERIA_PRIORITY.index(d["criterion"])
                      if d["criterion"] in CRITERIA_PRIORITY else 99
    )

print(f"Stories with actionable defects : {len(work_list)}")
print(f"Total defects to address        : {sum(len(v) for v in work_list.values())}")

human_input_needed = {
    k: [d for d in defs if d["criterion"] in HUMAN_INPUT_NEEDED]
    for k, defs in work_list.items()
    if any(d["criterion"] in HUMAN_INPUT_NEEDED for d in defs)
}
print(f"Stories needing your input      : {len(human_input_needed)}")
print()

# ── Q&A loop ──────────────────────────────────────────────────────────────
user_qa = {}   # (abstract_id, llm, story_id, criterion) → answer string

for (abstract_id, llm_name, story_id), defects in human_input_needed.items():
    gk = f"{abstract_id}_{llm_name}"

    # Get story text and group context
    story_rows = results_df[
        (results_df["group"] == gk) & (results_df["story_id"] == story_id)
    ]
    if story_rows.empty:
        continue
    story_text    = story_rows.iloc[0]["story_text"]
    group_stories = results_df[results_df["group"] == gk]["story_text"].tolist()

    print("─" * 70)
    print(f"Group  : {gk}")
    print(f"Story  : [{story_id}] {story_text}")
    print()

    for defect in defects:
        crit       = defect["criterion"]
        model_key  = get_responsible_model(crit, defect["votes"])
        qa_key     = (abstract_id, llm_name, story_id, crit)

        print(f"  Criterion : {crit.upper()}  (detected by: {model_key}, "
              f"conf={defect['confidence']:.2f})")
        print(f"  Generating clarification question via {model_key}...")

        question = agent_generate_question(crit, story_text, group_stories, model_key)

        print()
        print(f"  ❓ Question: {question}")
        answer = input("  Your answer: ").strip()
        user_qa[qa_key] = answer
        print()

print("─" * 70)
print(f"\n✓ Collected {len(user_qa)} answer(s). Run the next cell to rewrite.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 2 — REWRITE + RE-EVALUATE + SAVE
# ═══════════════════════════════════════════════════════════════════════════
# For every story in the work list:
#   1. Each defect criterion → responsible model rewrites
#   2. Output of one rewrite feeds the next criterion (sequential)
#   3. Responsible model self-checks: did the rewrite actually fix it?
#   4. atomic splits produce multiple output stories
# ═══════════════════════════════════════════════════════════════════════════

enhancement_log = []   # for final report

# Working copy so we can update story_text in results_df
results_df["enhanced"]      = False
results_df["enhanced_text"] = None

rows_to_add = []   # atomic splits that become new rows

for (abstract_id, llm_name, story_id), defects in work_list.items():
    gk = f"{abstract_id}_{llm_name}"

    story_rows = results_df[
        (results_df["group"] == gk) & (results_df["story_id"] == story_id)
    ]
    if story_rows.empty:
        continue
    idx        = story_rows.index[0]
    original   = story_rows.iloc[0]["story_text"]
    group_all  = results_df[results_df["group"] == gk]["story_text"].tolist()

    print(f"\n[{gk}] {story_id}")
    print(f"  Original: {original[:100]}...")

    current_texts = [original]   # may become list after atomic split
    log_entry = {
        "group": gk, "story_id": story_id, "original": original,
        "defects": [], "final": []
    }

    for defect in defects:
        crit      = defect["criterion"]
        model_key = get_responsible_model(crit, defect["votes"])
        qa_key    = (abstract_id, llm_name, story_id, crit)
        answer    = user_qa.get(qa_key, "")

        print(f"  ✏  [{crit}] → {model_key}"
              + (f" (with user clarification)" if answer else ""))

        # Build updated group context (use current rewritten text if available)
        group_ctx = [
            current_texts[0] if t == original else t
            for t in group_all
        ]

        new_texts = []
        for txt in current_texts:
            rewrites = agent_rewrite(crit, txt, group_ctx, model_key, answer)
            new_texts.extend(rewrites)

        if not new_texts:
            print(f"     → Removed as duplicate")
            current_texts = []
            break

        # Quick self-check per rewritten story
        checked = []
        for rt in new_texts:
            passed = agent_reeval(crit, rt, group_ctx, model_key)
            status = "✓ fixed" if passed else "⚠ still failing"
            print(f"     → {rt[:90]}...  [{status}]")
            checked.append({"text": rt, "passed": passed})

        current_texts = [c["text"] for c in checked]
        log_entry["defects"].append({
            "criterion": crit, "model": model_key,
            "user_answer": answer,
            "rewrites": [c["text"] for c in checked],
            "all_passed": all(c["passed"] for c in checked),
        })

        time.sleep(0.3)   # gentle rate limiting

    log_entry["final"] = current_texts
    enhancement_log.append(log_entry)

    if not current_texts:
        results_df.at[idx, "enhanced"]      = True
        results_df.at[idx, "enhanced_text"] = "[REMOVED — duplicate]"
        continue

    # First rewritten story replaces original in-place
    results_df.at[idx, "enhanced"]      = True
    results_df.at[idx, "enhanced_text"] = current_texts[0]

    # Any additional stories from an atomic split become new rows
    for j, extra in enumerate(current_texts[1:], start=1):
        new_row = results_df.loc[idx].copy()
        new_row["story_id"]      = f"{story_id}_s{j}"
        new_row["story_text"]    = extra
        new_row["enhanced"]      = True
        new_row["enhanced_text"] = extra
        rows_to_add.append(new_row)

if rows_to_add:
    results_df = pd.concat(
        [results_df, pd.DataFrame(rows_to_add)], ignore_index=True
    )

results_df["final_text"] = results_df.apply(
    lambda r: r["enhanced_text"]
              if r["enhanced"] and pd.notna(r.get("enhanced_text"))
              else r["story_text"],
    axis=1,
)

# ── Summary report ─────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ENHANCEMENT SUMMARY")
print("=" * 70)
n_enhanced = sum(1 for e in enhancement_log if e["final"])
n_removed  = sum(1 for e in enhancement_log if not e["final"])
n_splits   = sum(max(0, len(e["final"]) - 1) for e in enhancement_log)
n_qa_used  = sum(
    1 for e in enhancement_log
    for d in e["defects"] if d.get("user_answer")
)
print(f"  Stories rewritten  : {n_enhanced}")
print(f"  Stories removed    : {n_removed}  (exact duplicates)")
print(f"  Atomic splits      : {n_splits}   (extra stories added)")
print(f"  User answers used  : {n_qa_used}")

from collections import Counter
crit_counts   = Counter(d["criterion"] for e in enhancement_log for d in e["defects"])
fix_rates     = {}
for e in enhancement_log:
    for d in e["defects"]:
        c = d["criterion"]
        fix_rates.setdefault(c, []).append(d.get("all_passed", False))

print("\n  Per-criterion results:")
print(f"  {'Criterion':<22} {'Fixed':>7} {'Failed':>7} {'Model'}")
print("  " + "-" * 55)
for crit in CRITERIA_PRIORITY:
    if crit not in crit_counts: continue
    rates = fix_rates.get(crit, [])
    n_ok  = sum(rates)
    n_bad = len(rates) - n_ok
    # example responsible model (use first vote that fired)
    ex_votes = json.dumps({m: 1 for m in ENSEMBLE_RULES.get(crit, ("", ["deepseek"]))[1]})
    resp = get_responsible_model(crit, ex_votes) or "—"
    print(f"  {crit:<22} {n_ok:>7} {n_bad:>7}  {resp}")

# ── Side-by-side comparison CSV ────────────────────────────────────────────
comparison_rows = []
for e in enhancement_log:
    for d in e["defects"]:
        comparison_rows.append({
            "group":            e["group"],
            "story_id":         e["story_id"],
            "criterion":        d["criterion"],
            "responsible_model":d["model"],
            "original":         e["original"],
            "rewritten":        " | ".join(d["rewrites"]),
            "fix_verified":     d.get("all_passed", False),
            "user_answer":      d.get("user_answer", ""),
        })

comp_df = pd.DataFrame(comparison_rows) if comparison_rows else pd.DataFrame()

# ── Save ───────────────────────────────────────────────────────────────────
import zipfile, pathlib

results_df.to_csv("ensemble_enhanced.csv", index=False)
if not comp_df.empty:
    comp_df.to_csv("enhancement_comparison.csv", index=False)
with open("enhancement_log.json", "w") as f:
    json.dump(enhancement_log, f, indent=2, default=str)

ZIP = "/content/enhanced_output.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["ensemble_enhanced.csv", "enhancement_comparison.csv",
                  "enhancement_log.json", "ensemble_flags_all.csv",
                  "ensemble_labels.csv"]:
        p = pathlib.Path(f"/content/{fname}")
        if p.exists():
            zf.write(str(p), fname)
            print(f"  + {fname}")

print(f"\n✓ Saved to {ZIP}")
from google.colab import files
files.download(ZIP)
